# Benchmark Basic seafloor detection

Compare Echopype `main` with another Pull Request (here, against [PR #1805](https://github.com/echostack-org/echopype/pull/1805)) using the benchmark runner, across dataset sizes and Dask chunk configurations. This notebook uses the **`config.yaml`** and **`manifest.csv`** files in the folder. Echopype prepares the largest requested dataset once, or reuses existing prepared data when available.

> **Note:** Timings depend on hardware and the number of benchmark rounds. Output comparison is exact; here, `DIFFERENT` ouputs may reflect the PR's intended handling of missing detections.

## 1. Set up the runner

Use an existing Echopype checkout locally or clone the repository in Google Colab; first, we detect whether the notebook is running locally or in Google Colab:

In [ ]:
import importlib.util

IN_COLAB = (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
)

We set up Echopype and select the Git revisions to benchmark without modifying the local branch:

In [ ]:
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

UPSTREAM = "https://github.com/echostack-org/echopype.git"
BASE_REF = "main"
HEAD_REF = "refs/pull/1805/head"
BENCHMARK = "mask/seafloor_detection/benchmark_bottom_basic.py"

if IN_COLAB:
    REPO = Path("/content/echopype")

    if not REPO.exists():
        subprocess.run(
            ["git", "clone", UPSTREAM, str(REPO)],
            check=True,
        )

    subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
    subprocess.run(
        [
            "uv",
            "pip",
            "install",
            "--python",
            sys.executable,
            "-e",
            ".",
            "--group",
            "benchmark-test",
        ],
        cwd=REPO,
        check=True,
    )
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO = next(
        p / "echopype"
        for p in candidates
        if (p / "echopype/echopype/benchmarks/run.py").exists()
    )

RUNNER = REPO / "echopype/benchmarks/run.py"


def commit(ref):
    subprocess.run(["git", "fetch", UPSTREAM, ref], cwd=REPO, check=True)
    return subprocess.check_output(
        ["git", "rev-parse", "FETCH_HEAD"], cwd=REPO, text=True
    ).strip()


BASE, HEAD = commit(BASE_REF), commit(HEAD_REF)
print("Runner:", RUNNER)
print("Base:", BASE[:12], "→ Head:", HEAD[:12])

## 2. Review the experiment

We can edit **`config.yaml`** to choose file counts, backend profiles, chunk sizes and repetitions.

In [ ]:
import yaml

if IN_COLAB:
    EXPERIMENT = Path("/content")
    if not all(
        (EXPERIMENT / name).exists() for name in ("config.yaml", "manifest.csv")
    ):
        from google.colab import files

        files.upload()  # Select config.yaml and manifest.csv
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    EXPERIMENT = next(
        p
        for root in candidates
        for p in (root, root / "notebooks/benchmark/basic_seafloor")
        if (p / "config.yaml").exists() and (p / "manifest.csv").exists()
    )

CONFIG = EXPERIMENT / "config.yaml"
config = yaml.safe_load(CONFIG.read_text())
print(CONFIG.read_text())

## 3. Run the benchmark

Use a previously prepared Zarr dataset if available; otherwise Echopype prepares data once. The runner handles the revisions, backends, measurements and reports.

In [ ]:
DATA_DIR = REPO / "benchmark-results/compare-100-files/data"
if not (DATA_DIR / "ek80_hake.zarr").exists():
    DATA_DIR = None

OUTPUT = (
    REPO / "benchmark-results" / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
)
cmd = [
    sys.executable,
    "-u",
    str(RUNNER),
    "--base",
    BASE,
    "--head",
    HEAD,
    "--benchmark",
    BENCHMARK,
    "--config",
    str(CONFIG),
    "--output",
    str(OUTPUT),
]
if DATA_DIR is not None:
    cmd += ["--data-dir", str(DATA_DIR)]

print("Prepared data:", DATA_DIR or "prepare from manifest")
with subprocess.Popen(
    cmd,
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)

    if process.wait() != 0:
        raise subprocess.CalledProcessError(process.returncode, cmd)
print("Results:", OUTPUT)

## 4. Compare execution times

The table reports median execution time and speedup.

In [ ]:
import json
import pandas as pd
from IPython.display import display

rows = []
for case in sorted(OUTPUT.glob("*/*_files")):
    times = {}
    for ref in ("base", "head"):
        benchmarks = json.loads((case / f"{ref}.json").read_text())["benchmarks"]
        times[ref] = benchmarks[0]["stats"]["median"]
    rows.append(
        {
            "RAW files": int(case.name.split("_")[0]),
            "Configuration": case.parent.name,
            "Main (s)": times["base"],
            "PR #1805 (s)": times["head"],
            "Speedup": times["base"] / times["head"],
        }
    )

results = pd.DataFrame(rows).sort_values(["RAW files", "Configuration"])
display(results.round(2))

## 5. Inspect numerical agreement and echograms

The reports use `xr.testing.assert_identical()`. When outputs differ, the report marks them as `DIFFERENT`.

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown((OUTPUT / "summary.md").read_text()))
for figure in sorted(OUTPUT.glob("*/*_files/*.png")):
    display(Markdown(f"**{figure.parent.relative_to(OUTPUT)}**"))
    display(Image(filename=str(figure)))